# Automated VWAP V1 Backtest Runner

This notebook is a clean automated backtest runner for the VWAP Probability Band Engine project.

It tests one defined automated strategy version on historical OHLC CSV data.

The notebook is separate from the research notebook and separate from the visual/discretionary MT5 overlay.

This notebook does not connect to MT5 and does not place live trades.

## Strategy Version

**Version:** Automated VWAP V1

The current V1 candidate uses:

- continuation-only logic
- both longs and shorts allowed
- green reclaim / lower-green rejection structure
- weak directional red-shift filter
- entry at next bar open
- fixed 29-point stop loss
- fixed 58-point take profit
- breakeven after +29 points
- 1% risk per trade
- stop after 2 consecutive daily losses
- stop after +8% realised daily profit
- no new trades after 19:00 Europe/London

The aim is to make this notebook a clean V1 runner that can later be duplicated into V2, V3, V4, or converted into a `.py` backtest/live runner.

## 1. Project Setup

This section imports the required libraries, detects the project root, and defines the main project folders used by the notebook.

In [ ]:
from pathlib import Path
from pprint import pprint
import json

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start_path: Path | None = None) -> Path:
    """
    Find the project root by walking upward until a .git folder is found.

    This allows the notebook to work whether it is run from the project root
    or from inside the notebooks folder.
    """
    if start_path is None:
        start_path = Path.cwd()

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / ".git").exists():
            return path

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
HISTORICAL_DATA_DIR = DATA_DIR / "historical"
SRC_DIR = PROJECT_ROOT / "src"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = ARTIFACTS_DIR / "automated_vwap_v1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", Path.cwd())
print("Detected project root:", PROJECT_ROOT)
print("Historical data folder:", HISTORICAL_DATA_DIR)
print("Source folder:", SRC_DIR)
print("Output folder:", OUTPUT_DIR)

## 2. Editable Strategy Configuration

This is the main control panel for the V1 backtest.

Most basic V1 experiments should be possible by changing this cell only.

In [ ]:
CONFIG = {
    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "csv_file": "US100_cash_M1_NY_session_30d.csv",
    "dataset_name": "US100_cash_M1_NY_session_30d",

    # ------------------------------------------------------------------
    # Strategy identity
    # ------------------------------------------------------------------
    "strategy_version": "automated_vwap_v1",
    "strategy_description": "Continuation VWAP reclaim/rejection with weak directional red-shift filter",

    # ------------------------------------------------------------------
    # Session handling
    # ------------------------------------------------------------------
    "session_timezone": "Europe/London",
    "no_new_trades_after": "19:00",

    # ------------------------------------------------------------------
    # Direction controls
    # ------------------------------------------------------------------
    "allow_longs": True,
    "allow_shorts": True,

    # ------------------------------------------------------------------
    # Strategy filters
    # ------------------------------------------------------------------
    "strategy_family": "continuation_only",
    "strategy_filter": "weak_red_shift_only",
    "red_shift_filter_choice": "weak_directional",

    # ------------------------------------------------------------------
    # Entry logic
    # ------------------------------------------------------------------
    "entry_timing": "next_bar_open",

    # ------------------------------------------------------------------
    # Fixed Nasdaq point trade levels
    # ------------------------------------------------------------------
    "sl_points": 29.0,
    "tp_points": 58.0,
    "be_trigger_points": 29.0,

    # ------------------------------------------------------------------
    # Risk controls
    # ------------------------------------------------------------------
    "risk_per_trade_pct": 1.0,
    "daily_max_consecutive_losses": 2,
    "daily_profit_cap_pct": 8.0,

    # ------------------------------------------------------------------
    # Candle quality filters
    # ------------------------------------------------------------------
    "min_body_ratio": 0.30,
    "min_close_through_green": 0.0,
    "max_extension_from_green": 35.0,

    # ------------------------------------------------------------------
    # Output settings
    # ------------------------------------------------------------------
    "save_trade_log": True,
    "save_daily_summary": True,
    "save_skipped_signals": True,
    "save_config_snapshot": True,
}

pprint(CONFIG)

## 3. Trade Level Sanity Check

This section checks that the configured stop loss, take profit, and breakeven levels behave correctly for both long and short trades.

The config stores all point distances as positive numbers.

The trade simulator will later convert those distances into the correct long or short price levels.

In [ ]:
def build_trade_levels(entry_price: float, side: str, config: dict) -> dict:
    """
    Build fixed-point Nasdaq execution levels for a long or short trade.
    """
    side = side.lower()

    sl = float(config["sl_points"])
    tp = float(config["tp_points"])
    be = float(config["be_trigger_points"])

    if side == "long":
        return {
            "side": "long",
            "entry_price": entry_price,
            "stop_price": entry_price - sl,
            "breakeven_trigger_price": entry_price + be,
            "target_price": entry_price + tp,
        }

    if side == "short":
        return {
            "side": "short",
            "entry_price": entry_price,
            "stop_price": entry_price + sl,
            "breakeven_trigger_price": entry_price - be,
            "target_price": entry_price - tp,
        }

    raise ValueError("side must be either 'long' or 'short'")


example_entry = 20000.0

level_check = {
    "long_example": build_trade_levels(example_entry, "long", CONFIG),
    "short_example": build_trade_levels(example_entry, "short", CONFIG),
}

pprint(level_check)

## 4. Data Loading

This section loads the configured OHLC CSV file.

It handles common MT5-style column names and standardises them into:

- `datetime`
- `open`
- `high`
- `low`
- `close`

Optional fields such as `tick_volume`, `spread`, and `real_volume` are preserved when available.

In [ ]:
REQUIRED_RAW_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
]

COLUMN_ALIASES = {
    "datetime": [
        "datetime",
        "time",
        "timestamp",
        "date",
        "Date",
        "Time",
        "Datetime",
        "Local time",
        "Gmt time",
        "GMT time",
    ],
    "open": ["open", "Open", "OPEN"],
    "high": ["high", "High", "HIGH"],
    "low": ["low", "Low", "LOW"],
    "close": ["close", "Close", "CLOSE"],
    "tick_volume": [
        "tick_volume",
        "volume",
        "Volume",
        "tickvol",
        "Tick Volume",
        "Tick volume",
    ],
    "spread": ["spread", "Spread"],
    "real_volume": ["real_volume", "Real Volume", "real volume"],
}


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    Return the first matching column from a list of possible column names.
    """
    existing = set(df.columns)

    for col in candidates:
        if col in existing:
            return col

    lower_map = {str(col).lower(): col for col in df.columns}

    for col in candidates:
        if str(col).lower() in lower_map:
            return lower_map[str(col).lower()]

    return None


def standardise_raw_ohlc_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename common OHLC column names into the standard names used by this notebook.
    """
    out = df.copy()
    rename_map = {}

    for standard_name, aliases in COLUMN_ALIASES.items():
        matched_col = find_column(out, aliases)

        if matched_col is not None and matched_col != standard_name:
            rename_map[matched_col] = standard_name

    out = out.rename(columns=rename_map)

    return out


def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    """
    Validate that the dataframe contains the required columns.
    """
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )


def prepare_raw_ohlc_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare raw OHLC data for the automated V1 backtest.

    This function only cleans and validates raw market data.

    It does not calculate VWAP.
    It does not calculate probability bands.
    It does not change the existing VWAP engine logic.
    """
    out = standardise_raw_ohlc_columns(df)

    validate_required_columns(out, REQUIRED_RAW_COLUMNS)

    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out = out.dropna(subset=["datetime"]).copy()

    numeric_cols = ["open", "high", "low", "close"]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).copy()

    for optional_col in ["tick_volume", "spread", "real_volume"]:
        if optional_col in out.columns:
            out[optional_col] = pd.to_numeric(out[optional_col], errors="coerce")

    out = out.sort_values("datetime").reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    return out


def list_candidate_data_files() -> list[Path]:
    """
    Find likely CSV or Parquet files inside the project.
    """
    patterns = ["*.csv", "*.parquet"]
    ignored_parts = {
        ".git",
        ".venv",
        "venv",
        "__pycache__",
        ".ipynb_checkpoints",
    }

    files = []

    for pattern in patterns:
        files.extend(PROJECT_ROOT.rglob(pattern))

    clean_files = [
        file for file in files
        if not any(part in ignored_parts for part in file.parts)
    ]

    return sorted(set(clean_files))


def resolve_data_file(config: dict) -> Path:
    """
    Resolve the configured data file.

    The config can use:
    - a file name, e.g. US100_cash_M1_NY_session_30d.csv
    - a relative path, e.g. data/historical/file.csv
    - an absolute path
    """
    configured_file = Path(config["csv_file"])

    if configured_file.is_absolute() and configured_file.exists():
        return configured_file

    direct_project_path = PROJECT_ROOT / configured_file

    if direct_project_path.exists():
        return direct_project_path

    candidate_files = list_candidate_data_files()

    matching_files = [
        file for file in candidate_files
        if file.name == configured_file.name
    ]

    if len(matching_files) == 1:
        return matching_files[0]

    if len(matching_files) > 1:
        print("Multiple matching files found:")

        for file in matching_files:
            print("-", file.relative_to(PROJECT_ROOT))

        raise ValueError(
            f"Multiple files named {configured_file.name} were found. "
            "Use a more specific relative path in CONFIG['csv_file']."
        )

    print(f"Could not find configured file: {config['csv_file']}")
    print("\nAvailable candidate files:")

    for file in candidate_files[:100]:
        print("-", file.relative_to(PROJECT_ROOT))

    raise FileNotFoundError(
        f"Could not find configured data file: {config['csv_file']}"
    )


def load_market_data(config: dict) -> tuple[pd.DataFrame, Path]:
    """
    Load the configured CSV or Parquet file and return a cleaned OHLC dataframe.
    """
    data_file = resolve_data_file(config)

    if data_file.suffix.lower() == ".csv":
        raw_df = pd.read_csv(data_file)
    elif data_file.suffix.lower() == ".parquet":
        raw_df = pd.read_parquet(data_file)
    else:
        raise ValueError(f"Unsupported file type: {data_file.suffix}")

    prepared_df = prepare_raw_ohlc_dataframe(raw_df)

    return prepared_df, data_file

In [ ]:
raw_ohlc_df, DATA_FILE = load_market_data(CONFIG)

print(f"Loaded data file: {DATA_FILE.relative_to(PROJECT_ROOT)}")
print(f"Rows loaded: {len(raw_ohlc_df):,}")
print(f"Start datetime: {raw_ohlc_df['datetime'].min()}")
print(f"End datetime: {raw_ohlc_df['datetime'].max()}")

print("\nColumns:")
print(list(raw_ohlc_df.columns))

print("\nFirst 5 rows:")
print(raw_ohlc_df.head().to_string(index=False))

## 5. Dataset Summary

This section prints a compact summary of the loaded dataset before the VWAP engine is applied.

In [ ]:
def summarise_raw_ohlc_data(df: pd.DataFrame, config: dict, data_file: Path) -> dict:
    """
    Create a compact summary of the loaded OHLC dataset.
    """
    summary = {
        "dataset_name": config["dataset_name"],
        "configured_file": config["csv_file"],
        "resolved_file": str(data_file.relative_to(PROJECT_ROOT)),
        "rows": len(df),
        "start_datetime": df["datetime"].min(),
        "end_datetime": df["datetime"].max(),
        "open_min": df["open"].min(),
        "open_max": df["open"].max(),
        "high_max": df["high"].max(),
        "low_min": df["low"].min(),
        "close_min": df["close"].min(),
        "close_max": df["close"].max(),
        "mean_candle_range": df["candle_range"].mean(),
        "median_candle_range": df["candle_range"].median(),
        "mean_body_ratio": df["body_ratio"].mean(),
        "zero_range_candles": int((df["candle_range"] <= 0).sum()),
        "duplicate_datetimes": int(df["datetime"].duplicated().sum()),
    }

    return summary


dataset_summary = summarise_raw_ohlc_data(raw_ohlc_df, CONFIG, DATA_FILE)

pprint(dataset_summary)

## 6. Ready for VWAP Engine Integration

At this point, the notebook has:

- loaded the configured dataset
- standardised OHLC column names
- parsed timestamps
- sorted candles chronologically
- created basic candle features
- printed a dataset summary

The next section will pass `raw_ohlc_df` into the existing VWAP Probability Band Engine from `src/`.

In [ ]:
print("raw_ohlc_df is ready for VWAP engine integration.")
print(f"Shape: {raw_ohlc_df.shape}")

print("\nLast 5 rows:")
print(raw_ohlc_df.tail().to_string(index=False))